In [9]:
# !pip install timm pandas scikit-learn tqdm matplotlib pillow torchvision
!pip install kagglehub

  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 270.8 kB/s eta 0:00:0000:01
Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl (324 kB)


In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pranabr0y/celebdf-v2image-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1


In [11]:
import os

def show_tree(root, max_depth=2):
    root = os.path.abspath(root)
    for current_root, dirs, files in os.walk(root):
        depth = current_root.replace(root, "").count(os.sep)
        if depth > max_depth:
            continue
        indent = "    " * depth
        print(f"{indent}{os.path.basename(current_root)}/")
        for f in files[:5]:
            print(f"{indent}    {f}")
        if len(files) > 5:
            print(f"{indent}    ... {len(files) - 5} more files")

show_tree(path, max_depth=3)

1/
    Celeb_V2/
        Test/
            fake/
                id0_id16_0000_face_6.jpg
                id0_id16_0001_face_13.jpg
                id0_id16_0002_face_18.jpg
                id0_id16_0002_face_26.jpg
                id0_id16_0004_face_44.jpg
                ... 5062 more files
            real/
                00000_face_14.jpg
                00000_face_2.jpg
                00000_face_25.jpg
                00000_face_32.jpg
                00000_face_49.jpg
                ... 5031 more files
        Train/
            fake/
                id0_id16_0000_face_0.jpg
                id0_id16_0000_face_1.jpg
                id0_id16_0000_face_2.jpg
                id0_id16_0000_face_3.jpg
                id0_id16_0000_face_4.jpg
                ... 40531 more files
            real/
                00000_face_0.jpg
                00000_face_1.jpg
                00000_face_10.jpg
                00000_face_11.jpg
                00000_face_12.jpg
                ... 40

In [12]:
from pathlib import Path
import pandas as pd

root = Path(path) / "Celeb_V2"
print(root)

/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2


In [13]:
def build_csv_from_split(split_path):
    rows = []

    real_dir = split_path / "real"
    fake_dir = split_path / "fake"

    for img_path in real_dir.glob("*"):
        if img_path.is_file():
            rows.append({
                "image_path": str(img_path),
                "label": 0
            })

    for img_path in fake_dir.glob("*"):
        if img_path.is_file():
            rows.append({
                "image_path": str(img_path),
                "label": 1
            })

    return pd.DataFrame(rows)

In [14]:
train_df = build_csv_from_split(root / "Train")
val_df = build_csv_from_split(root / "Val")
test_df = build_csv_from_split(root / "Test")

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Train size: 80824
Val size: 10104
Test size: 10103


In [15]:
print("Train label counts")
print(train_df["label"].value_counts())

print("\nVal label counts")
print(val_df["label"].value_counts())

print("\nTest label counts")
print(test_df["label"].value_counts())

Train label counts
label
1    40536
0    40288
Name: count, dtype: int64

Val label counts
label
1    5068
0    5036
Name: count, dtype: int64

Test label counts
label
1    5067
0    5036
Name: count, dtype: int64


In [37]:
print("Using DataFrames directly")
print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Using DataFrames directly
Train size: 80824
Val size: 10104
Test size: 10103


In [38]:
import os
import time
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from torchvision import transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

In [39]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [40]:
from dataclasses import dataclass
from pathlib import Path
import os

@dataclass
class Config:
    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 2
    epochs: int = 10
    lr: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.2
    mixed_precision: bool = True

    backbone_name: str = "tf_efficientnet_b4.ns_jft_in1k"

    uncertain_low: float = 0.4
    uncertain_high: float = 0.6

    out_dir: str = str(Path.cwd() / "outputs")
    ckpt_name: str = "best_model.pt"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)
print("Output dir:", cfg.out_dir)

Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs


In [41]:
def get_transforms(image_size=224, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))],
                p=0.15
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
            transforms.RandomErasing(
                p=0.25,
                scale=(0.02, 0.08),
                ratio=(0.3, 3.3),
                value="random"
            ),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])


In [42]:
class CelebDFBinaryDataset(Dataset):
    def __init__(self, dataframe, image_size=224, train=True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = get_transforms(image_size=image_size, train=train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["image_path"]
        label = float(row["label"])

        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Could not read image: {path}") from e

        img = self.transform(img)

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.float32),
            "path": path,
        }


def make_loader(dataframe, image_size, batch_size, num_workers, train):
    ds = CelebDFBinaryDataset(dataframe=dataframe, image_size=image_size, train=train)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=train,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=train,
    )


def make_loaders(config, train_df, val_df, test_df):
    train_loader = make_loader(train_df, config.image_size, config.batch_size, config.num_workers, True)
    val_loader = make_loader(val_df, config.image_size, config.batch_size, config.num_workers, False)
    test_loader = make_loader(test_df, config.image_size, config.batch_size, config.num_workers, False)
    return train_loader, val_loader, test_loader

In [43]:
class BinaryDeepfakeDetector(nn.Module):
    def __init__(self, backbone_name="tf_efficientnet_b4.ns_jft_in1k", dropout=0.2):
        super().__init__()
        self.encoder = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 1),
        )

    def forward(self, x):
        feat = self.encoder(x)
        logits = self.head(feat).squeeze(1)
        return logits


In [44]:
def compute_metrics(y_true, y_prob, uncertain_low=0.4, uncertain_high=0.6):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    uncertain_rate = ((y_prob > uncertain_low) & (y_prob < uncertain_high)).mean() * 100.0

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
    }


In [45]:
def train_one_epoch(model, loader, optimizer, scaler, config):
    model.train()
    running_loss = 0.0
    y_true_all = []
    y_prob_all = []

    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels = y.detach().cpu().numpy()

        y_true_all.extend(labels.tolist())
        y_prob_all.extend(probs.tolist())
        running_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    metrics = compute_metrics(y_true_all, y_prob_all, config.uncertain_low, config.uncertain_high)
    metrics["loss"] = running_loss / len(loader)
    return metrics


@torch.no_grad()
def validate_one_epoch(model, loader, config):
    model.eval()
    running_loss = 0.0
    y_true_all = []
    y_prob_all = []

    for batch in tqdm(loader, desc="val", leave=False):
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        logits = model(x)
        loss = F.binary_cross_entropy_with_logits(logits, y)

        probs = torch.sigmoid(logits).cpu().numpy()
        labels = y.cpu().numpy()

        y_true_all.extend(labels.tolist())
        y_prob_all.extend(probs.tolist())
        running_loss += loss.item()

    metrics = compute_metrics(y_true_all, y_prob_all, config.uncertain_low, config.uncertain_high)
    metrics["loss"] = running_loss / len(loader)
    return metrics


In [46]:
def fit_model(config, train_df, val_df, test_df):
    train_loader, val_loader, test_loader = make_loaders(config, train_df, val_df, test_df)

    model = BinaryDeepfakeDetector(config.backbone_name, config.dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(config.mixed_precision and device.type == "cuda"))

    history = []
    best_val_auc = -1.0
    train_start = time.time()

    for epoch in range(1, config.epochs + 1):
        epoch_start = time.time()

        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, config)
        val_metrics = validate_one_epoch(model, val_loader, config)
        epoch_mins = (time.time() - epoch_start) / 60.0

        row = {
            "Epoch": epoch,
            "Train Loss": train_metrics["loss"],
            "Train Acc": train_metrics["accuracy"],
            "Train AUC": train_metrics["auc"],
            "Val Acc": val_metrics["accuracy"],
            "Val AUC": val_metrics["auc"],
            "Precision": val_metrics["precision"],
            "Recall": val_metrics["recall"],
            "F1": val_metrics["f1"],
            "Uncertain Rate (%)": val_metrics["uncertain_rate"],
            "Epoch mins": epoch_mins,
        }
        history.append(row)
        history_df = pd.DataFrame(history)

        print(
            f"Epoch {epoch:02d}/{config.epochs} | "
            f"Train Loss {train_metrics['loss']:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Train AUC {train_metrics['auc']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Val AUC {val_metrics['auc']:.4f} | "
            f"Precision {val_metrics['precision']:.4f} | "
            f"Recall {val_metrics['recall']:.4f} | "
            f"F1 {val_metrics['f1']:.4f} | "
            f"Uncertain {val_metrics['uncertain_rate']:.2f}%"
        )

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            torch.save(
                {
                    "model": model.state_dict(),
                    "config": config.__dict__,
                    "best_val_auc": best_val_auc,
                },
                os.path.join(config.out_dir, config.ckpt_name),
            )
            print("saved best checkpoint")

    total_train_mins = (time.time() - train_start) / 60.0
    print(f"total training time: {total_train_mins:.2f} mins | {total_train_mins / 60.0:.2f} hrs")

    ckpt = torch.load(os.path.join(config.out_dir, config.ckpt_name), map_location=device)
    model.load_state_dict(ckpt["model"])

    return model, history_df, train_loader, val_loader, test_loader, total_train_mins

In [47]:
@torch.no_grad()
def evaluate_model(model, loader, config, model_name="EfficientNet-B4"):
    model.eval()
    y_true_all = []
    y_prob_all = []

    for batch in tqdm(loader, desc="test", leave=False):
        x = batch["image"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        labels = y.cpu().numpy()

        y_true_all.extend(labels.tolist())
        y_prob_all.extend(probs.tolist())

    m = compute_metrics(y_true_all, y_prob_all, config.uncertain_low, config.uncertain_high)
    return pd.DataFrame([{
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "AUC": m["auc"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Uncertain Rate (%)": m["uncertain_rate"],
    }])


In [48]:
@torch.no_grad()
def evaluate_resolutions(model, test_df, batch_size, num_workers, config, resolutions=(128, 224, 256, 384)):
    model.eval()
    rows = []

    for res in resolutions:
        loader = make_loader(test_df, res, batch_size, num_workers, train=False)
        y_true_all = []
        y_prob_all = []

        for batch in tqdm(loader, desc=f"res {res}", leave=False):
            x = batch["image"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)

            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy()
            labels = y.cpu().numpy()

            y_true_all.extend(labels.tolist())
            y_prob_all.extend(probs.tolist())

        m = compute_metrics(y_true_all, y_prob_all, config.uncertain_low, config.uncertain_high)
        rows.append({
            "Resolution": f"{res}x{res}",
            "Accuracy": m["accuracy"],
            "AUC": m["auc"],
        })

    return pd.DataFrame(rows)

In [49]:
@torch.no_grad()
def measure_inference_time_by_resolution(model, test_df, batch_size, num_workers, resolutions=(128, 224, 256, 384)):
    rows = []
    for res in resolutions:
        loader = make_loader(test_df, res, batch_size, num_workers, train=False)
        df = measure_inference_time(model, loader, mode_name=f"{res}x{res}")
        rows.append(df.iloc[0].to_dict())
    return pd.DataFrame(rows)

In [50]:
def make_training_time_table(model_type, total_train_mins):
    return pd.DataFrame([{
        "Model Type": model_type,
        "mins": total_train_mins,
        "hrs": total_train_mins / 60.0,
    }])

In [51]:
def plot_resolution_results(res_df):
    plot_df = res_df.copy()
    plot_df["res_int"] = plot_df["Resolution"].str.split("x").str[0].astype(int)
    plot_df = plot_df.sort_values("res_int")

    plt.figure(figsize=(7, 4))
    plt.plot(plot_df["res_int"], plot_df["Accuracy"], marker="o", label="Accuracy")
    plt.plot(plot_df["res_int"], plot_df["AUC"], marker="s", label="AUC")
    plt.xlabel("Resolution")
    plt.ylabel("Score")
    plt.title("Performance vs Resolution")
    plt.legend()
    plt.grid(True)
    plt.show()

In [52]:
cfg.image_size = 224

model, history_df, train_loader, val_loader, test_loader, total_train_mins = fit_model(
    cfg, train_df, val_df, test_df
)

/tmp/ipykernel_124/3423721712.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(config.mixed_precision and device.type == "cuda"))
train:   0%|          | 0/5051 [00:00<?, ?it/s]/tmp/ipykernel_124/841755354.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):


Epoch 01/10 | Train Loss 0.1077 | Train Acc 0.9562 | Train AUC 0.9929 | Val Acc 0.9914 | Val AUC 0.9998 | Precision 0.9978 | Recall 0.9850 | F1 0.9914 | Uncertain 0.42%
saved best checkpoint


train:   0%|          | 0/5051 [00:00<?, ?it/s]/tmp/ipykernel_124/841755354.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):


Epoch 02/10 | Train Loss 0.0326 | Train Acc 0.9884 | Train AUC 0.9993 | Val Acc 0.9936 | Val AUC 0.9999 | Precision 0.9978 | Recall 0.9893 | F1 0.9936 | Uncertain 0.27%
saved best checkpoint


train:   0%|          | 0/5051 [00:00<?, ?it/s]/tmp/ipykernel_124/841755354.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):


...

Epoch 08/10 | Train Loss 0.0115 | Train Acc 0.9960 | Train AUC 0.9999 | Val Acc 0.9975 | Val AUC 1.0000 | Precision 0.9986 | Recall 0.9964 | F1 0.9975 | Uncertain 0.11%
saved best checkpoint


train:   0%|          | 0/5051 [00:00<?, ?it/s]/tmp/ipykernel_124/841755354.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):


Epoch 09/10 | Train Loss 0.0092 | Train Acc 0.9967 | Train AUC 0.9999 | Val Acc 0.9972 | Val AUC 1.0000 | Precision 0.9949 | Recall 0.9996 | F1 0.9972 | Uncertain 0.16%


train:   0%|          | 0/5051 [00:00<?, ?it/s]/tmp/ipykernel_124/841755354.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(config.mixed_precision and device.type == "cuda")):


Epoch 10/10 | Train Loss 0.0086 | Train Acc 0.9970 | Train AUC 0.9999 | Val Acc 0.9960 | Val AUC 1.0000 | Precision 0.9927 | Recall 0.9994 | F1 0.9961 | Uncertain 0.21%
total training time: 70.11 mins | 1.17 hrs


In [53]:
test_results_df = evaluate_model(
    model,
    test_loader,
    cfg,
    model_name="EfficientNet-B4 Real/Fake"
)

test_results_df

,Model,Accuracy,AUC,Precision,Recall,F1,Uncertain Rate (%)
0,EfficientNet-B4 Real/Fake,0.996833,0.999936,0.999009,0.994671,0.996835,0.079184


In [54]:
res_df = evaluate_resolutions(
    model,
    test_df=test_df,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    config=cfg,
    resolutions=(128, 224, 256, 384),
)

res_df

,Resolution,Accuracy,AUC
0,128x128,0.832624,0.946703
1,224x224,0.996833,0.999936
2,256x256,0.996932,0.999912
3,384x384,0.966149,0.997199


In [55]:
inf_df = measure_inference_time_by_resolution(
    model,
    test_df=test_df,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    resolutions=(128, 224, 256, 384),
)

inf_df

,Mode,Time per Image (ms),FPS
0,128x128,0.943721,1059.634946
1,224x224,1.342483,744.888630
2,256x256,1.242658,804.726707
3,384x384,3.009890,332.238086


In [56]:
train_time_df = make_training_time_table(
    "EfficientNet-B4 Real/Fake",
    total_train_mins
)

train_time_df

,Model Type,mins,hrs
0,EfficientNet-B4 Real/Fake,70.110487,1.168508


In [57]:
from pathlib import Path
import os

save_dir = Path.cwd() / "outputs"
os.makedirs(save_dir, exist_ok=True)

history_df.to_csv(save_dir / "training_period_results.csv", index=False)
test_results_df.to_csv(save_dir / "test_results.csv", index=False)
res_df.to_csv(save_dir / "accuracy_based_on_resolution.csv", index=False)
inf_df.to_csv(save_dir / "inference_time.csv", index=False)
train_time_df.to_csv(save_dir / "training_time.csv", index=False)

print("Saved to:", save_dir)

Saved to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs
